In [1]:
import os
import sys

# 현재 작업 디렉토리 확인해보고
print("CWD:", os.getcwd())

# notebooks/ 에서 한 단계 위(프로젝트 루트)로 올라가기
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("PROJECT_ROOT:", PROJECT_ROOT)

# sys.path 에 프로젝트 루트가 없으면 추가
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print("Added to sys.path")


CWD: c:\Users\user\Desktop\project\scalp-vision-agent\notebooks
PROJECT_ROOT: c:\Users\user\Desktop\project\scalp-vision-agent
Added to sys.path


In [2]:
import torch
from torch.utils.data import DataLoader
from torch import optim

from src.config import MASTER_INDEX_CSV
from src.cnn.dataset import ScalpDataset
from src.cnn.models import MultiHeadResNet18
from src.cnn.utils import labels_dict_to_tensor
from src.cnn.losses import multihead_ce_loss
from src.cnn.train import get_device, train_one_epoch, evaluate_one_epoch, train_model

from src.config import PROJECT_ROOT


In [3]:
device = get_device()
device


device(type='cuda')

In [4]:
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4


In [ ]:
import torchvision.transforms as T


train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
    ),
    T.ToTensor(),
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])


In [6]:
from torch.utils.data import Subset

train_dataset = ScalpDataset(index_csv=MASTER_INDEX_CSV, split="train", transforms=train_transform)
val_dataset   = ScalpDataset(index_csv=MASTER_INDEX_CSV, split="val",   transforms=val_transform)

print(len(train_dataset), len(val_dataset))


67588 23568


In [7]:
USE_SUBSET = False   # overfit 테스트: True, 전체 학습: False

if USE_SUBSET:
    subset_indices = list(range(100))  
    train_dataset_small = Subset(train_dataset, subset_indices)
    train_loader = DataLoader(train_dataset_small, batch_size=BATCH_SIZE, shuffle=True)
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [8]:
model = MultiHeadResNet18()
model.to(device)

from torch.optim import AdamW
optimizer = AdamW(
    model.parameters(),
    lr=LR,           # 기존 lr 그대로 (너가 쓰던 값)
    weight_decay=1e-4, # 새로 추가
)


In [9]:
batch = next(iter(train_loader))

print("type:", type(batch["image"]))
print("shape:", batch["image"].shape)
print("dtype:", batch["image"].dtype)
print("device:", batch["image"].device)


type: <class 'torch.Tensor'>
shape: torch.Size([32, 3, 224, 224])
dtype: torch.float32
device: cpu


In [10]:
len(train_dataset), len(train_loader), len(val_dataset), len(val_loader)


(67588, 2113, 23568, 737)

In [11]:
# 셀 1: 기본 상수와 import

import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader

# 멀티헤드 이름 (이미 위에서 정의했어도 다시 정의해도 무방)
HEAD_NAMES = ["value_1", "value_2", "value_3", "value_4", "value_5", "value_6"]

NUM_HEADS = len(HEAD_NAMES)  # 6
NUM_CLASSES = 4              # 각 head는 0~3 등급


In [12]:
# 셀 2: DataLoader 배치에서 image, labels 텐서만 안전하게 추출하는 함수

from typing import Any, Tuple

def extract_images_and_labels(batch: Any) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    DataLoader가 반환하는 batch에서
    - images: [B, 3, 224, 224] 텐서
    - labels: [B, NUM_HEADS] 텐서
    를 추출해주는 유틸 함수.

    batch는 dict 또는 (images, labels, ...) 튜플일 수 있다.
    labels가 텐서 혹은 dict일 수 있는 상황을 모두 처리한다.
    """
    # 1) batch 타입에 따른 분기
    if isinstance(batch, dict):
        # 케이스 A: dict 배치
        if "image" not in batch:
            raise KeyError("batch dict에 'image' 키가 없습니다.")
        images = batch["image"]

        if "labels" not in batch:
            # 예외적으로 labels 키가 없으면 첫 두 개 텐서를 image/labels로 가정
            tensor_values = [v for v in batch.values() if isinstance(v, torch.Tensor)]
            if len(tensor_values) < 2:
                raise ValueError("batch dict에서 labels 텐서를 찾을 수 없습니다.")
            labels_obj = tensor_values[1]
        else:
            labels_obj = batch["labels"]

    elif isinstance(batch, (list, tuple)):
        # 케이스 B: (images, labels, ...) 튜플
        if len(batch) < 2:
            raise ValueError("batch 튜플 길이가 2 미만입니다. images, labels를 둘 다 포함해야 합니다.")
        images, labels_obj = batch[0], batch[1]

    else:
        raise TypeError(f"지원하지 않는 batch 타입입니다: {type(batch)}")

    if not isinstance(images, torch.Tensor):
        raise TypeError(f"images가 torch.Tensor가 아닙니다: {type(images)}")

    # 2) labels_obj 타입에 따른 처리
    if isinstance(labels_obj, torch.Tensor):
        # 이미 [B, NUM_HEADS] 형태라고 가정
        labels = labels_obj
    elif isinstance(labels_obj, dict):
        # 예: {"value_1": tensor([..]), ..., "value_6": tensor([..])}
        value_keys = [k for k in labels_obj.keys() if k.startswith("value_")]
        if value_keys:
            # "value_1"~"value_6" 순으로 정렬
            value_keys = sorted(value_keys, key=lambda k: int(k.split("_")[1]))
            tensors = [labels_obj[k] for k in value_keys]
        else:
            # 혹시 value_ 키가 없다면 그냥 키 이름으로 정렬
            tensors = [labels_obj[k] for k in sorted(labels_obj.keys())]

        # 각 텐서는 [B] 라고 가정하고, [NUM_HEADS, B] → [B, NUM_HEADS] 로 스택
        tensors = [t if t.ndim == 1 else t.view(-1) for t in tensors]
        labels = torch.stack(tensors, dim=1)
    else:
        raise TypeError(f"labels_obj가 텐서도 dict도 아닙니다: {type(labels_obj)}")

    # 최종적으로 [B, NUM_HEADS]인지 한 번 더 체크
    if labels.ndim != 2 or labels.size(1) != NUM_HEADS:
        raise ValueError(
            f"labels의 shape가 예상과 다릅니다: {labels.shape}, "
            f"기대한 형상은 [B, {NUM_HEADS}] 입니다."
        )

    return images, labels


In [13]:
# 셀 3: train_loader를 한 번 훑어서 각 head별 class count 계산

head_class_counts = torch.zeros((NUM_HEADS, NUM_CLASSES), dtype=torch.long)

model.eval()
with torch.no_grad():
    for batch in train_loader:
        images, labels = extract_images_and_labels(batch)  # labels: [B, 6]
        labels = labels.long()

        for h in range(NUM_HEADS):
            head_labels = labels[:, h]  # [B]
            for c in range(NUM_CLASSES):
                head_class_counts[h, c] += (head_labels == c).sum()

head_class_counts


tensor([[55609,  4435,  5457,  2087],
        [13970, 26538, 23584,  3496],
        [20658, 29811, 12874,  4245],
        [64482,  2124,   702,   280],
        [41495, 15617,  8565,  1911],
        [51312, 12185,  3402,   689]])

In [14]:
# 셀 4: head_class_counts를 바탕으로 inverse-frequency 기반 class weight 계산

# 0인 곳이 있다면 division by zero를 막기 위해 epsilon 더해줌
eps = 1e-6

# 각 head별 총 샘플 수: [NUM_HEADS, 1]
head_totals = head_class_counts.sum(dim=1, keepdim=True).float() + eps

# class frequency: 각 head별 [NUM_HEADS, NUM_CLASSES]
class_freq = head_class_counts.float() / head_totals

# inverse frequency: 자주 나오는 class는 작게, 희소한 class는 크게
inv_freq = 1.0 / (class_freq + eps)

# head별로 평균이 1이 되도록 정규화 (scale 안정화)
head_class_weights = inv_freq / inv_freq.mean(dim=1, keepdim=True)

head_class_weights


tensor([[0.0794, 0.9956, 0.8092, 2.1158],
        [0.6542, 0.3444, 0.3875, 2.6140],
        [0.4900, 0.3395, 0.7862, 2.3843],
        [0.0113, 0.3436, 1.0394, 2.6057],
        [0.1324, 0.3518, 0.6414, 2.8745],
        [0.0422, 0.1778, 0.6367, 3.1434]])

In [15]:
# 셀 5: 멀티헤드 weighted CE loss + accuracy 함수

def multihead_weighted_ce_loss(
    logits: torch.Tensor,        # [B, NUM_HEADS, NUM_CLASSES]
    targets: torch.Tensor,       # [B, NUM_HEADS]
    class_weights: torch.Tensor, # [NUM_HEADS, NUM_CLASSES]
) -> torch.Tensor:
    """
    head마다 다른 class weight를 적용하는 멀티헤드 CE loss.

    - logits[h]에 대해 class_weights[h]를 weight로 사용
    - head별 loss 평균을 최종 loss로 사용
    """
    B, H, C = logits.shape
    assert H == NUM_HEADS and C == NUM_CLASSES, f"Unexpected logits shape: {logits.shape}"

    loss = 0.0
    for h in range(H):
        weight_h = class_weights[h].to(logits.device)
        loss_h = F.cross_entropy(
            logits[:, h, :],      # [B, C]
            targets[:, h],        # [B]
            weight=weight_h,      # [C]
        )
        loss = loss + loss_h

    loss = loss / H
    return loss


def compute_multihead_accuracy(
    logits: torch.Tensor,   # [B, NUM_HEADS, NUM_CLASSES]
    targets: torch.Tensor,  # [B, NUM_HEADS]
) -> float:
    """
    멀티헤드 전체에 대해 단순 평균 정확도 계산.
    (모든 head / 모든 샘플 기준으로 정확도 평균)
    """
    preds = logits.argmax(dim=-1)  # [B, NUM_HEADS]
    acc = (preds == targets).float().mean().item()
    return acc


In [16]:
# 셀 6: 검증용 evaluate 함수 (전체 loss/acc + head별 loss/acc)

def evaluate_one_epoch_with_heads(
    model: torch.nn.Module,
    dataloader: DataLoader,
    device: torch.device,
) -> tuple[float, float, list[float], list[float]]:
    """
    - 전체 val_loss, val_acc
    - head별 loss/acc 리스트까지 함께 반환
    """
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    head_losses = torch.zeros(NUM_HEADS, dtype=torch.float32)
    head_accs = torch.zeros(NUM_HEADS, dtype=torch.float32)

    with torch.no_grad():
        for batch in dataloader:
            images, labels = extract_images_and_labels(batch)
            images = images.to(device)
            labels = labels.long().to(device)  # [B, NUM_HEADS]

            logits = model(images)  # [B, NUM_HEADS, NUM_CLASSES]

            # 멀티헤드 CE (unweighted)로 평가
            B, H, C = logits.shape
            loss = 0.0
            for h in range(H):
                loss_h = F.cross_entropy(logits[:, h, :], labels[:, h])
                loss = loss + loss_h
            loss = loss / H

            batch_acc = compute_multihead_accuracy(logits, labels)

            total_loss += loss.item()
            total_acc += batch_acc
            n_batches += 1

            # head별 loss/acc 계산
            for h in range(H):
                head_logits = logits[:, h, :]
                head_targets = labels[:, h]
                loss_h = F.cross_entropy(head_logits, head_targets)
                preds_h = head_logits.argmax(dim=-1)
                acc_h = (preds_h == head_targets).float().mean().item()

                head_losses[h] += loss_h.item()
                head_accs[h] += acc_h

    avg_loss = total_loss / n_batches
    avg_acc = total_acc / n_batches

    head_losses /= n_batches
    head_accs /= n_batches

    return avg_loss, avg_acc, head_losses.tolist(), head_accs.tolist()


# -----------------------------------------------

In [17]:
# 셀 1: EarlyStopping 유틸 + import

from __future__ import annotations

import copy
from dataclasses import dataclass
from typing import Any

import torch


@dataclass
class EarlyStopping:
    """val_loss 기준 Early Stopping 헬퍼."""
    patience: int = 3
    min_delta: float = 0.0  # 개선으로 인정할 최소 감소량

    best: float | None = None
    num_bad_epochs: int = 0

    def step(self, current: float) -> bool:
        """
        val_loss를 넣어서 호출.
        True를 리턴하면 early stopping 조건을 만족했다는 의미.
        """
        if self.best is None or current < self.best - self.min_delta:
            self.best = current
            self.num_bad_epochs = 0
            return False

        self.num_bad_epochs += 1
        return self.num_bad_epochs >= self.patience


In [18]:
# 셀 2: E4 실험용 하이퍼파라미터 & 스케줄러

num_epochs = 20  # 최대 epoch (early stopping으로 실제는 더 일찍 끝날 수 있음)

early_stopper = EarlyStopping(
    patience=3,     # val_loss가 3 epoch 연속 개선 안 되면 멈춤
    min_delta=1e-4, # 아주 미세한 흔들림은 무시
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",     # val_loss가 줄어드는 방향이 좋음
    factor=0.5,     # 개선 없을 때 lr을 절반으로 줄임
    patience=1,     # 1 epoch 동안 개선 없으면 lr 감소
    verbose=True,
)

# 로그 저장용
history: dict[str, list] = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_value6_acc": [],   # value_6 전용 검증 정확도 기록
    "val_head_losses": [],  # 각 head별 loss(딕셔너리)를 매 epoch 저장
    "val_head_accs": [],    # 각 head별 acc(딕셔너리)를 매 epoch 저장
}

best_val_loss = float("inf")
best_value6_acc = 0.0

best_state_by_loss: dict[str, Any] | None = None
best_state_by_value6: dict[str, Any] | None = None


c:\Users\user\Desktop\project\scalp-vision-agent\venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [19]:
EXPERIMENT_NAME = "E4_class_weight_earlystop"

# project_root/results/E4_class_weight_earlystop/
results_root = PROJECT_ROOT / "results"
results_dir = results_root / EXPERIMENT_NAME
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Results will be saved under: {results_dir.resolve()}")

Results will be saved under: C:\Users\user\Desktop\project\scalp-vision-agent\results\E4_class_weight_earlystop


In [21]:
import json

In [22]:
# E4 메인 학습 루프 (list → dict 변환 포함)

HEAD_NAMES = ["value_1", "value_2", "value_3", "value_4", "value_5", "value_6"]

for epoch in range(1, num_epochs + 1):
    print(f"\n===== [E4] Epoch {epoch} / {num_epochs} =====")

    # 1) Train
    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        device=device,
    )

    # 2) Validation
    #   여기서 raw_* 는 list 또는 dict일 수 있다고 보고 처리
    val_loss, val_acc, raw_head_losses, raw_head_accs = (
        evaluate_one_epoch_with_heads(
            model=model,
            dataloader=val_loader,
            device=device,
        )
    )

    # 2-1) head별 결과를 dict 형태로 통일
    if isinstance(raw_head_losses, dict) and isinstance(raw_head_accs, dict):
        val_head_losses = {k: float(v) for k, v in raw_head_losses.items()}
        val_head_accs = {k: float(v) for k, v in raw_head_accs.items()}
    elif isinstance(raw_head_losses, (list, tuple)) and isinstance(raw_head_accs, (list, tuple)):
        # HEAD_NAMES 순서대로 매핑
        val_head_losses = {
            name: float(v) for name, v in zip(HEAD_NAMES, raw_head_losses)
        }
        val_head_accs = {
            name: float(v) for name, v in zip(HEAD_NAMES, raw_head_accs)
        }
    else:
        raise TypeError(
            f"Unexpected types for head metrics: "
            f"{type(raw_head_losses)}, {type(raw_head_accs)}"
        )

    # value_6(탈모) 정확도만 따로 추출
    value6_acc = float(val_head_accs.get("value_6", 0.0))

    # 3) 로그 출력
    print(
        f"[Summary] train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, "
        f"value_6_acc={value6_acc:.4f}"
    )
    print("[Val per-head metrics]")
    for k in sorted(val_head_accs.keys()):
        print(
            f" - {k}: loss={val_head_losses[k]:.4f}, acc={val_head_accs[k]:.4f}"
        )

    # 4) history 업데이트
    history["epoch"].append(epoch)
    history["train_loss"].append(float(train_loss))
    history["train_acc"].append(float(train_acc))
    history["val_loss"].append(float(val_loss))
    history["val_acc"].append(float(val_acc))
    history["val_value6_acc"].append(value6_acc)
    history["val_head_losses"].append(val_head_losses)
    history["val_head_accs"].append(val_head_accs)

    # 5) LR scheduler
    scheduler.step(val_loss)

    # 6) best-by-loss
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = float(val_loss)
        best_state_by_loss = copy.deepcopy(model.state_dict())
        print(f"  ↳ Best (val_loss) model updated! val_loss={best_val_loss:.4f}")

    # 7) best-by-value6
    if value6_acc > best_value6_acc + 1e-4:
        best_value6_acc = value6_acc
        best_state_by_value6 = copy.deepcopy(model.state_dict())
        print(
            f"  ↳ Best (value_6) model updated! value_6_acc={best_value6_acc:.4f}"
        )

    # 8) history 중간 저장
    history_path = results_dir / "history.json"
    with history_path.open("w", encoding="utf-8") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)
    print(f"  ↳ Saved history snapshot to {history_path.name}")

    # 9) Early stopping
    if early_stopper.step(val_loss):
        print(
            f"\n🛑 Early stopping triggered at epoch {epoch} "
            f"(best_val_loss={early_stopper.best:.4f})"
        )
        break

print("\n===== [E4] Training finished =====")
print(f"Best val_loss: {best_val_loss:.4f}")
print(f"Best value_6_acc: {best_value6_acc:.4f}")



===== [E4] Epoch 1 / 20 =====
[Summary] train_loss=0.6232, train_acc=0.7475 | val_loss=0.5941, val_acc=0.7557, value_6_acc=0.7386
[Val per-head metrics]
 - value_1: loss=0.4472, acc=0.8554
 - value_2: loss=0.9553, acc=0.5735
 - value_3: loss=0.7089, acc=0.6956
 - value_4: loss=0.1930, acc=0.9530
 - value_5: loss=0.6549, acc=0.7183
 - value_6: loss=0.6051, acc=0.7386
  ↳ Best (val_loss) model updated! val_loss=0.5941
  ↳ Best (value_6) model updated! value_6_acc=0.7386
  ↳ Saved history snapshot to history.json

===== [E4] Epoch 2 / 20 =====
[Summary] train_loss=0.6029, train_acc=0.7540 | val_loss=0.6543, val_acc=0.7255, value_6_acc=0.7219
[Val per-head metrics]
 - value_1: loss=0.4609, acc=0.8546
 - value_2: loss=0.9667, acc=0.5555
 - value_3: loss=0.8980, acc=0.6112
 - value_4: loss=0.1941, acc=0.9525
 - value_5: loss=0.7786, acc=0.6574
 - value_6: loss=0.6275, acc=0.7219
  ↳ Saved history snapshot to history.json

===== [E4] Epoch 3 / 20 =====
[Summary] train_loss=0.5902, train_acc=

In [24]:
import pandas as pd

In [25]:
# 셀 4: 체크포인트 & 표 저장

# 1) 모델 가중치 저장
ckpt_by_loss = results_dir / "best_by_val_loss.pt"
ckpt_by_value6 = results_dir / "best_by_value6_acc.pt"
ckpt_last = results_dir / "last_epoch.pt"

# 마지막 epoch 상태도 같이 저장해두고 싶으면:
torch.save(model.state_dict(), ckpt_last)
print(f"✔ Saved last-epoch model to {ckpt_last.name}")

if best_state_by_loss is not None:
    torch.save(best_state_by_loss, ckpt_by_loss)
    print(f"✔ Saved best-by-loss model to {ckpt_by_loss.name}")

if best_state_by_value6 is not None:
    torch.save(best_state_by_value6, ckpt_by_value6)
    print(f"✔ Saved best-by-value6 model to {ckpt_by_value6.name}")

# 2) history를 표(전체 지표)로 저장
#   - head별 dict 리스트는 따로 풀어서 저장

metrics_df = pd.DataFrame({
    "epoch": history["epoch"],
    "train_loss": history["train_loss"],
    "train_acc": history["train_acc"],
    "val_loss": history["val_loss"],
    "val_acc": history["val_acc"],
    "val_value6_acc": history["val_value6_acc"],
})
metrics_csv = results_dir / "metrics_overall.csv"
metrics_df.to_csv(metrics_csv, index=False)
print(f"✔ Saved overall metrics table to {metrics_csv.name}")

# 3) head별 val_acc를 wide 형태 테이블로 변환해서 저장
#    (epoch x head_name)
head_acc_rows: list[dict[str, float]] = []
for ep, head_acc in zip(history["epoch"], history["val_head_accs"]):
    row = {"epoch": ep}
    row.update(head_acc)
    head_acc_rows.append(row)

head_acc_df = pd.DataFrame(head_acc_rows).set_index("epoch")
head_acc_csv = results_dir / "metrics_val_head_acc.csv"
head_acc_df.to_csv(head_acc_csv)
print(f"✔ Saved per-head val_acc table to {head_acc_csv.name}")


✔ Saved last-epoch model to last_epoch.pt
✔ Saved best-by-loss model to best_by_val_loss.pt
✔ Saved best-by-value6 model to best_by_value6_acc.pt
✔ Saved overall metrics table to metrics_overall.csv
✔ Saved per-head val_acc table to metrics_val_head_acc.csv


In [26]:
import matplotlib.pyplot as plt
# matplotlib 한글 폰트 설정 (필요시)
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows 기준
# plt.rcParams['font.family'] = 'AppleGothic' # Mac 기준
plt.rcParams['axes.unicode_minus'] = False # 음수 부호 깨짐 방지

In [27]:
# 셀 5: 학습 곡선 & head별 val_acc 시각화

epochs = history["epoch"]

# 1) Loss 곡선
plt.figure(figsize=(6, 4))
plt.plot(epochs, history["train_loss"], label="train_loss")
plt.plot(epochs, history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("E4: Train vs Val Loss")
plt.legend()
plt.tight_layout()
loss_png = results_dir / "plot_loss.png"
plt.savefig(loss_png)
plt.close()
print(f"✔ Saved {loss_png.name}")

# 2) Accuracy 곡선 (전체 + value_6)
plt.figure(figsize=(6, 4))
plt.plot(epochs, history["train_acc"], label="train_acc")
plt.plot(epochs, history["val_acc"], label="val_acc")
plt.plot(epochs, history["val_value6_acc"], label="val_value6_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("E4: Train/Val/Value6 Accuracy")
plt.legend()
plt.tight_layout()
acc_png = results_dir / "plot_acc.png"
plt.savefig(acc_png)
plt.close()
print(f"✔ Saved {acc_png.name}")

# 3) head별 val_acc 곡선
#   - history["val_head_accs"]는 epoch마다 dict[str, float]
all_head_names: set[str] = set()
for head_acc in history["val_head_accs"]:
    all_head_names.update(head_acc.keys())
head_names = sorted(all_head_names)

plt.figure(figsize=(6, 4))
for head in head_names:
    vals: list[float] = []
    for head_acc in history["val_head_accs"]:
        vals.append(float(head_acc.get(head, float("nan"))))
    plt.plot(epochs, vals, marker="o", label=head)

plt.xlabel("Epoch")
plt.ylabel("Val Accuracy")
plt.title("E4: Per-head Validation Accuracy")
plt.legend()
plt.tight_layout()
head_acc_png = results_dir / "plot_val_head_acc.png"
plt.savefig(head_acc_png)
plt.close()
print(f"✔ Saved {head_acc_png.name}")


✔ Saved plot_loss.png
✔ Saved plot_acc.png
✔ Saved plot_val_head_acc.png


# -----------------------------------------------------------------